<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [3]:

// Интерфейс для отправки чека
public interface IReceiptable
{
    void SendReceipt(string email);
}

// Интерфейс для возврата средств
public interface IRefundable
{
    void Refund(decimal amount);
}

// Интерфейс для логирования транзакций (будет реализован явно)
public interface ITransactionLogger
{
    void LogTransaction(string info);
    string GetLastLog();
}

// Базовый класс
public class PaymentMethod : ITransactionLogger
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission;
    private decimal _maxAmount;
    private bool _isActive;
    private DateTime _lastTransactionDate;

    private string _transactionId;
    private int _transactionCount;
    private string _lastLog;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    public string Currency { get; set; }

    public decimal MaxAmount
    {
        get { return _maxAmount; }
        set { _maxAmount = value; }
    }

    public bool IsActive
    {
        get { return _isActive; }
        set { _isActive = value; }
    }

    public DateTime LastTransactionDate
    {
        get { return _lastTransactionDate; }
        set { _lastTransactionDate = value; }
    }

    public string TransactionId
    {
        get { return _transactionId; }
        set { _transactionId = value; }
    }

    public int TransactionCount
    {
        get { return _transactionCount; }
        set { _transactionCount = value; }
    }

    public PaymentMethod(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, decimal maxAmount = 1000000m)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
        Currency = "RUB";
        MaxAmount = maxAmount;
        IsActive = true;
        _transactionCount = 0;
    }

    public bool CheckMinimumAmount(decimal amount) => amount >= MinAmount;
    public bool CheckMaximumAmount(decimal amount) => amount <= MaxAmount;
    public decimal CalculateCommission(decimal amount) => amount * (Commission / 100);

    public void CancelPayment()
    {
        Console.WriteLine($"Платеж по способу '{MethodName}' отменен.");
    }

    public void GenerateTransactionId()
    {
        TransactionId = $"{PaymentMethodId}-{DateTime.Now.Ticks}";
        Console.WriteLine($"Сгенерирован ID транзакции: {TransactionId}");
    }

    // ИЗМЕНЕНО: теперь возвращает bool, чтобы наследники знали, прошла ли валидация
    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"Способ оплаты '{MethodName}' временно неактивен.");
            return false;
        }

        if (!CheckMaximumAmount(amount))
        {
            Console.WriteLine($"Ошибка: сумма {amount} {Currency} превышает максимальную ({MaxAmount} {Currency}).");
            return false;
        }

        if (CheckMinimumAmount(amount))
        {
            decimal commission = CalculateCommission(amount);
            LastTransactionDate = DateTime.Now;
            _transactionCount++;
            GenerateTransactionId();

            Console.WriteLine($"Успешно: платеж на сумму {amount} {Currency} (комиссия: {commission} {Currency}) проведен через '{MethodName}'.");
            ((ITransactionLogger)this).LogTransaction($"Платеж {amount} {Currency} через {MethodName}");
            return true;
        }
        else
        {
            Console.WriteLine($"Ошибка: сумма {amount} {Currency} меньше минимальной ({MinAmount} {Currency}) для '{MethodName}'.");
            return false;
        }
    }

    // ИЗМЕНЕНО: комментарий выводится только при успешном платеже
    public bool ProcessPayment(decimal amount, string comment)
    {
        bool success = ProcessPayment(amount);
        if (success)
        {
            Console.WriteLine($"Комментарий к платежу: {comment}");
        }
        return success;
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine(
            $"ID способа: {PaymentMethodId} | Название: {MethodName} | " +
            $"Мин. сумма: {MinAmount} {Currency} | Макс. сумма: {MaxAmount} {Currency} | " +
            $"Комиссия: {Commission}% | Активен: {IsActive} | Транзакций: {TransactionCount}");
    }

    // Явная реализация интерфейса
    void ITransactionLogger.LogTransaction(string info)
    {
        _lastLog = $"[{DateTime.Now:HH:mm:ss}] {info}";
        Console.WriteLine($"Лог записан: {_lastLog}");
    }

    string ITransactionLogger.GetLastLog()
    {
        return _lastLog ?? "Логов пока нет.";
    }
}

// Онлайн-оплата
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    public OnlinePayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
    }

    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }

    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"URL для оплаты: {PaymentUrl}");
    }
}

// Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;
    private string _bic;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    public string Bic
    {
        get { return _bic; }
        set { _bic = value; }
    }

    public BankTransfer(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string bankData, string bic = "044525225")
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
        Bic = bic;
    }

    // ИЗМЕНЕНО: реквизиты выводятся ТОЛЬКО при успешной валидации в базе
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Перевод отправлен по реквизитам: {BankData}, БИК: {Bic}");
        return true;
    }

    public void VerifyBankDetails()
    {
        Console.WriteLine($"Проверка реквизитов: {BankData} / БИК {Bic} — ОК.");
    }
}

// Оплата наличными
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;
    private TimeSpan _workingHours;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    public TimeSpan WorkingHours
    {
        get { return _workingHours; }
        set { _workingHours = value; }
    }

    public CashPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
        WorkingHours = new TimeSpan(10, 0, 0);
    }

    // ИЗМЕНЕНО: точка самовывоза выводится ТОЛЬКО при успехе
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Ожидайте оплату наличными в точке: {CashPickupPoint}");
        return true;
    }

    public void CheckAvailability()
    {
        Console.WriteLine($"Точка {CashPickupPoint} работает {WorkingHours.Hours} часов в день.");
    }
}

// Оплата банковской картой
public class CardPayment : OnlinePayment, IRefundable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }
    public DateTime CardExpiry { get; set; }

    public CardPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl,
        string cardNumber, string cardHolder, DateTime? cardExpiry = null)
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
        CardExpiry = cardExpiry ?? DateTime.Now.AddYears(3);
    }

    public void Refund(decimal amount)
    {
        Console.WriteLine($"Оформлен возврат {amount} {Currency} на карту {CardNumber} ({CardHolder})");
    }

    public void Refund(decimal amount, string reason)
    {
        Refund(amount);
        Console.WriteLine($"Причина возврата: {reason}");
    }

    // ИЗМЕНЕНО: списание с карты происходит ТОЛЬКО при успешной валидации
    public override bool ProcessPayment(decimal amount)
    {
        if (!base.ProcessPayment(amount))
            return false;

        Console.WriteLine($"Списание с карты {CardNumber}, держатель: {CardHolder}");
        return true;
    }

    public bool IsCardExpired() => DateTime.Now > CardExpiry;
}

// Generic класс
public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _payments = new List<T>();

    public void Add(T payment) => _payments.Add(payment);

    public void ProcessAll(decimal amount)
    {
        foreach (var payment in _payments)
        {
            payment.ProcessPayment(amount);
        }
    }

    public void ShowAllDetails()
    {
        foreach (var payment in _payments)
        {
            payment.GetPaymentDetails();
        }
    }
}

// Класс заказа
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    // ИЗМЕНЕНО: теперь заказ знает, оплачен он или нет
    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine($"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");

        bool isSuccess = paymentMethod.ProcessPayment(TotalAmount);
        if (isSuccess)
        {
            Console.WriteLine($"Статус: Заказ №{OrderId} успешно оплачен.");
        }
        else
        {
            Console.WriteLine($"Статус: Ошибка при оплате заказа №{OrderId}.");
        }
    }
}

// Управление зависимостями (DI)
public class PaymentProcessorService
{
    private readonly PaymentMethod _paymentMethod;

    public PaymentProcessorService(PaymentMethod paymentMethod)
    {
        _paymentMethod = paymentMethod;
    }

    public void Execute(decimal amount)
    {
        _paymentMethod.GetPaymentDetails();
        bool success = _paymentMethod.ProcessPayment(amount);

        if (success)
        {
            ITransactionLogger logger = _paymentMethod;
            Console.WriteLine($"Последний лог: {logger.GetLastLog()}");
        }
    }
}